# Fig. 14, Schrödinger sphere: every computation of the figure in one notebook

Phyein Usul (phyeinusul.com). Data: dataset `numakoudsie/fig14-schrodinger-sphere` (attach it as input). Record of origin: `public/data/schrodinger-sphere/generations.json` in the site's repository. Built by `tools/kaggle/figures/build.py`.

Set `JOB` in the first code cell. `check` (default, CPU) verifies every stored file (size and SHA-256 against its manifest) and that every job's code compiles. Any other name reruns that job with the exact code that made the data; that code fetches its solver from the repository at the pinned commit. GPU jobs need the GPU switched on.

| JOB | computes | machine | commit | code and outputs in the dataset |
|---|---|---|---|---|
| `full-grid` | the sphere in 3D: 9 energies x 9 heights x 5 radii = 405 runs, 128^3 cells | GPU T4 x2 | `10dc1d6` | `schrodinger-sphere-full-grid/` |
| `3d-wave` | the 3D wave Re psi for the 3D view of the sphere | GPU T4 x2 | `7083b64` | `schrodinger-sphere-3d-wave/` |
| `flat-wall` | the flat wall (1D across the wall, exact in 3D) | CPU | `76c7ce0` | `schrodinger-flat-wall/` |
| `pillar` | the pillar (2D in the plane, exact in 3D): 585 runs | CPU | `89f6e9d` | `schrodinger-pillar/` |
| `spherical-wave` | a spherical wave onto the sphere (axisymmetric) | CPU | `13dda4c` | `schrodinger-spherical-wave/` |
| `shapes-3d` | coin, tube, wells, hard ball, spherical wave from the side (version 2: hard and shell runs, 243) | GPU T4 x2 | `94d1281` | `schrodinger-shapes-3d/` |
| `3d-generation` | a 3D cloud of 81 runs; on no page, kept as data only | GPU T4 x2 | `5941f63` | `schrodinger-sphere-3d-generation/` |

In [ ]:
JOB = "check"   # or one of: full-grid, 3d-wave, flat-wall, pillar, spherical-wave, shapes-3d, 3d-generation
DATASET = "numakoudsie/fig14-schrodinger-sphere"
JOBS = [
 {
  "name": "full-grid",
  "folder": "schrodinger-sphere-full-grid",
  "machine": "GPU T4 x2",
  "commit": "10dc1d64200446165d40456972d9b51047846f71",
  "what": "the sphere in 3D: 9 energies x 9 heights x 5 radii = 405 runs, 128^3 cells"
 },
 {
  "name": "3d-wave",
  "folder": "schrodinger-sphere-3d-wave",
  "machine": "GPU T4 x2",
  "commit": "7083b64c90303ffabe4355e7359a347a392a6485",
  "what": "the 3D wave Re psi for the 3D view of the sphere"
 },
 {
  "name": "flat-wall",
  "folder": "schrodinger-flat-wall",
  "machine": "CPU",
  "commit": "76c7ce089557d9939ed1f63f5535021057f387ee",
  "what": "the flat wall (1D across the wall, exact in 3D)"
 },
 {
  "name": "pillar",
  "folder": "schrodinger-pillar",
  "machine": "CPU",
  "commit": "89f6e9d3305f243b2507ae64f6ab9c4a186bdbef",
  "what": "the pillar (2D in the plane, exact in 3D): 585 runs"
 },
 {
  "name": "spherical-wave",
  "folder": "schrodinger-spherical-wave",
  "machine": "CPU",
  "commit": "13dda4cef3aeb6cd00ea3dbb48ee8a770344afcc",
  "what": "a spherical wave onto the sphere (axisymmetric)"
 },
 {
  "name": "shapes-3d",
  "folder": "schrodinger-shapes-3d",
  "machine": "GPU T4 x2",
  "commit": "94d1281585c91001fd6024c67b1045ed01691817",
  "what": "coin, tube, wells, hard ball, spherical wave from the side (version 2: hard and shell runs, 243)"
 },
 {
  "name": "3d-generation",
  "folder": "schrodinger-sphere-3d-generation",
  "machine": "GPU T4 x2",
  "commit": "5941f633a4c2a96a146e08c86bd3e5341e21c02a",
  "what": "a 3D cloud of 81 runs; on no page, kept as data only"
 }
]


In [ ]:
# ---- find the figure's dataset, then check it or run one job ----
import os, json, glob, hashlib, time, datetime
T0 = time.time()
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/README.md", recursive=True)
         if all(os.path.isdir(os.path.join(os.path.dirname(p), j["folder"])) for j in JOBS)]
assert cands, "attach the dataset %s (Add Input) and run again" % DATASET
DS = cands[0]; print("dataset at", DS)
def source_of(job):
    nb = json.load(open(os.path.join(DS, job["folder"], "source-current-version.ipynb")))
    return [("".join(c["source"]) if isinstance(c["source"], list) else c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
OUT = "/kaggle/working/" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
os.makedirs(OUT, exist_ok=True)
if JOB == "check":
    from IPython.core.inputtransformer2 import TransformerManager
    TM = TransformerManager(); report = dict(dataset=DATASET, path=DS, jobs={}, versions={})
    for j in JOBS:
        cells = source_of(j)
        for n, src in enumerate(cells): compile(TM.transform_cell(src), f"{j['name']} cell {n}", "exec")
        assert any(j["commit"] in src for src in cells), f"job {j['name']}: the stored code does not name commit {j['commit']}"
        report["jobs"][j["name"]] = dict(folder=j["folder"], code_cells=len(cells), compiles=True, commit_in_code=True)
        print(f"job {j['name']:15s} code: {len(cells)} cells, all compile, commit {j['commit'][:7]} named in the code")
    for man in sorted(glob.glob(os.path.join(DS, "*", "v*", "manifest.json"))):
        m = json.load(open(man)); base = os.path.join(os.path.dirname(man), "output")
        same = differ = unpacked = dropped = 0; missing = []
        for f in m["files"]:
            p = os.path.join(base, f["file"])
            if os.path.isfile(p):
                h = hashlib.sha256()
                with open(p, "rb") as fh:
                    for blk in iter(lambda: fh.read(1 << 22), b""): h.update(blk)
                ok = os.path.getsize(p) == f["bytes"] and h.hexdigest() == f["sha256"]
                same += ok; differ += not ok
            elif f["file"].endswith(".tar") and (os.path.isdir(p) or os.path.isdir(p[:-4])): unpacked += 1   # Kaggle unpacks tars
            elif f["file"].endswith(".pyc"): dropped += 1                                                   # Kaggle drops __pycache__
            else: missing.append(f["file"])
        key = os.path.relpath(os.path.dirname(man), DS)
        report["versions"][key] = dict(files=len(m["files"]), identical=same, different=differ, tar_unpacked=unpacked, pyc_dropped=dropped, missing=missing)
        print(f"{key:45s} {len(m['files']):5d} files: {same} identical, {differ} different, {unpacked} tars unpacked, {dropped} .pyc dropped, {len(missing)} missing")
    report["ok"] = all(v["different"] == 0 and not v["missing"] for v in report["versions"].values())
    report["seconds"] = round(time.time() - T0, 1)
    json.dump(report, open(OUT + "/check.json", "w"), indent=1)
    print("ALL OK" if report["ok"] else "PROBLEMS FOUND", "in", report["seconds"], "s")
else:
    job = next(j for j in JOBS if j["name"] == JOB)
    print("running job", JOB, "with the code of", job["folder"], "(commit", job["commit"] + ")")
    for n, src in enumerate(source_of(job)):
        print(f"---- cell {n} ----\n{src}")
        r = get_ipython().run_cell(src)
        if not r.success: raise RuntimeError(f"cell {n} of job {JOB} failed")
